# Chapter 21 — DBSCAN

**Density-Based Spatial Clustering of Applications with Noise**

DBSCAN is an unsupervised, density-based clustering algorithm that can identify arbitrary-shaped clusters and noise points.

## 1. Learning Objectives

- Understand density-based clustering
- Understand `eps` and `min_samples`
- Identify core, border, and noise points
- Apply DBSCAN using scikit-learn
- Experiment with DBSCAN parameters
- Use a k-distance plot to help select `eps`
- Interpret clustering results

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN

## 2. Create and Inspect the Dataset

A two-moons dataset is used because its curved structure demonstrates an important DBSCAN advantage over centroid-based clustering methods such as K-Means.

In [ ]:
from sklearn.datasets import make_moons

X, y = make_moons(
    n_samples=500,
    noise=0.08,
    random_state=42
)

df = pd.DataFrame(X, columns=['Feature_1', 'Feature_2'])
df.head()

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(df['Feature_1'], df['Feature_2'], s=40)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Raw Dataset — Two Moon-Shaped Clusters')
plt.show()

## 3. Feature Scaling

DBSCAN relies on distances between points, so feature scaling is important when features have different ranges.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Original shape:', X.shape)
print('Scaled shape:', X_scaled.shape)

pd.DataFrame(X_scaled, columns=['Feature_1', 'Feature_2']).head()

## 4. Apply DBSCAN

`eps` defines the neighborhood radius, while `min_samples` defines the minimum number of nearby points needed for a dense region. A label of `-1` represents noise.

In [ ]:
dbscan = DBSCAN(
    eps=0.3,
    min_samples=5
)

labels = dbscan.fit_predict(X_scaled)

print('Cluster labels:', np.unique(labels))

In [ ]:
n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = list(labels).count(-1)

print('Number of clusters:', n_clusters)
print('Number of noise points:', n_noise)

## 5. Visualize DBSCAN Clusters

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(X_scaled[:, 0], X_scaled[:, 1], c=labels, s=40)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('DBSCAN Clustering Result')
plt.show()

## 6. Identify Noise Points

DBSCAN uses the label `-1` for points that do not belong to any density-connected cluster.

In [ ]:
noise_points = X_scaled[labels == -1]
print('Number of noise points:', len(noise_points))

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(X_scaled[:, 0], X_scaled[:, 1], c=labels, s=40)

if len(noise_points) > 0:
    plt.scatter(
        noise_points[:, 0],
        noise_points[:, 1],
        s=80,
        marker='x',
        label='Noise'
    )
    plt.legend()

plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('DBSCAN — Clusters and Noise Points')
plt.show()

## 7. Experiment with `eps`

A smaller `eps` creates a stricter neighborhood, which can increase the number of noise points. A larger `eps` creates wider neighborhoods and may cause separate clusters to merge.

In [ ]:
eps_values = [0.15, 0.2, 0.25, 0.3, 0.4, 0.5]

for eps in eps_values:
    model = DBSCAN(eps=eps, min_samples=5)
    test_labels = model.fit_predict(X_scaled)

    clusters = len(set(test_labels)) - (1 if -1 in test_labels else 0)
    noise = list(test_labels).count(-1)

    print(f'eps={eps}: clusters={clusters}, noise={noise}')

## 8. Experiment with `min_samples`

In [ ]:
min_samples_values = [3, 5, 8, 10, 15]

for min_samples in min_samples_values:
    model = DBSCAN(eps=0.3, min_samples=min_samples)
    test_labels = model.fit_predict(X_scaled)

    clusters = len(set(test_labels)) - (1 if -1 in test_labels else 0)
    noise = list(test_labels).count(-1)

    print(f'min_samples={min_samples}: clusters={clusters}, noise={noise}')

## 9. Final DBSCAN Model

In [ ]:
final_dbscan = DBSCAN(eps=0.3, min_samples=5)
final_labels = final_dbscan.fit_predict(X_scaled)

final_clusters = len(set(final_labels)) - (1 if -1 in final_labels else 0)
final_noise = list(final_labels).count(-1)

print('Final number of clusters:', final_clusters)
print('Final number of noise points:', final_noise)

## 10. K-Distance Plot

The k-distance plot helps identify a reasonable `eps` value. The approximate elbow of the sorted neighbor-distance curve can be used as a candidate threshold.

In [ ]:
from sklearn.neighbors import NearestNeighbors

k = 5
neighbors = NearestNeighbors(n_neighbors=k)
neighbors.fit(X_scaled)

distances, indices = neighbors.kneighbors(X_scaled)
k_distances = np.sort(distances[:, -1])

plt.figure(figsize=(8, 6))
plt.plot(k_distances)
plt.xlabel('Data Points (Sorted)')
plt.ylabel('5th Nearest Neighbor Distance')
plt.title('K-Distance Plot for Choosing eps')
plt.grid()
plt.show()

## 11. Selected `eps` and Final Visualization

In [ ]:
eps = 0.3
model = DBSCAN(eps=eps, min_samples=5)
labels = model.fit_predict(X_scaled)

clusters = len(set(labels)) - (1 if -1 in labels else 0)
noise = list(labels).count(-1)

print('Selected eps:', eps)
print('Clusters:', clusters)
print('Noise points:', noise)

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(X_scaled[:, 0], X_scaled[:, 1], c=labels, s=40)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Final DBSCAN Clustering')
plt.show()

## 12. Conclusion

DBSCAN is a density-based clustering algorithm that groups data points according to local density rather than distance from a centroid.

In this practical, DBSCAN identified two non-spherical moon-shaped clusters without being given the number of clusters beforehand. The experiments also demonstrated how `eps` and `min_samples` influence clustering and how a k-distance plot can help choose `eps`.

### Key Concepts
- `eps` — neighborhood radius
- `min_samples` — minimum density requirement
- Core point
- Border point
- Noise point (`-1`)
- Density-based clustering
- K-distance plot